# SmartCap - صور: ANN مقابل CNN (من جوّه VS Code)

الـ notebook ده بيعمل كل حاجة بدون PowerShell: تنزيل/ترتيب الداتا ← تقسيم **80% train / 20% validation** ← تدريب **ANN** وتدريب **CNN** على نفس التقسيم ← **accuracy و loss و الوقت** ← اختبار على **صورك انت**.

**قبل ما تبدأ (مرة واحدة):**
1. افتح فولدر `smartcap_ai` في VS Code (File ▸ Open Folder).
2. اضغط **Select Kernel** فوق يمين ▸ **Python Environments** ▸ اختار `.venv313`.
3. شغّل الخلايا بالترتيب (Shift+Enter).

In [ ]:
# 1) setup: packages + folders  (run once)
import os, sys, subprocess
if not os.path.exists("src/train_img.py"):
    raise SystemExit("Open the smartcap_ai folder in VS Code (File > Open Folder), so this notebook runs inside it.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "matplotlib", "scikit-learn", "opencv-python", "kagglehub"], check=True)
sys.path.insert(0, os.path.abspath("src"))
import numpy as np, matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display
import torch
print("python", sys.version.split()[0], "| torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (CPU - slower but works)")

## 2) الداتا الجاهزة (FL3D من Kaggle)

جرّب التنزيل التلقائي. لو طلب تسجيل دخول Kaggle أو فشل، نزّل الملف يدوي من صفحة الداتا (Download) وفكّه في `data\fl3d_raw`.

In [ ]:
# 2a) try the automatic download (needs internet; Kaggle may ask for a login/token)
RAW = "data/fl3d_raw"
try:
    import kagglehub
    RAW = kagglehub.dataset_download("matjazmuc/frame-level-driver-drowsiness-detection-fl3d")
    print("downloaded to:", RAW)
except Exception as e:
    print("Automatic download did not work:", type(e).__name__, e)
    print("Download it manually from the Kaggle page, unzip into  data/fl3d_raw  and continue.")

In [ ]:
# 2b) what is inside? (look at folders / file names / csv)
!{sys.executable} src/prepare_dataset.py --src "{RAW}" --inspect

In [ ]:
# 2c) organise into dataset_prepared/<class>/*.jpg  (alert / microsleep / yawning).
# If the matched counts look wrong, paste the 2b output to Claude. Use --max-per-class 3000 for a quicker first run.
!{sys.executable} src/prepare_dataset.py --src "{RAW}" --dst dataset_prepared

## 3) تحميل الصور + تقسيم 80/20

`IMG` حجم الصورة (64 سريع، 96 أدق)، `GRAY=True` يخلّي الصور رمادي (أسرع). `EPOCHS` عدد مرات المرور على الداتا.

In [ ]:
import time, json
from pathlib import Path
import imgmodels as im, train_img as ti

DATA, IMG, GRAY, EPOCHS, BATCH, SEED = "dataset_prepared", 64, False, 20, 64, 0
MAX_PER_CLASS = 6000          # None = all ~53,000 images (slower, needs ~1 GB RAM). Start small.

t0 = time.perf_counter()
X, y, classes, paths = im.load_folder(DATA, IMG, GRAY, max_per_class=MAX_PER_CLASS, seed=SEED)
print(f"{len(X)} images | classes {classes} | per class {np.bincount(y).tolist()} | loaded in {time.perf_counter()-t0:.0f} s")
tr, va, by_video = ti.split_indices(paths, y, 0.2, SEED)      # 80% / 20%, by VIDEO when names are video__frame.jpg
print(f"split {'BY VIDEO (no frame of a validation video is in training)' if by_video else 'random'}")
print(f"train {len(tr)} (80%) | validation {len(va)} (20%) | validation per class {np.bincount(y[va], minlength=len(classes)).tolist()}")

## 4) ANN (شبكة fully-connected عادية)

In [ ]:
res_ann, model_ann = ti.run_experiment("ann", X, y, classes, tr, va, IMG, GRAY, EPOCHS, BATCH, 1e-3, SEED,
                                       out_dir="reports/img", model_dir="models")
display(Image("reports/img/ann_curves.png")); display(Image("reports/img/ann_confusion.png"))

## 5) CNN (شبكة تلافيفية)

In [ ]:
res_cnn, model_cnn = ti.run_experiment("cnn", X, y, classes, tr, va, IMG, GRAY, EPOCHS, BATCH, 1e-3, SEED,
                                       out_dir="reports/img", model_dir="models")
display(Image("reports/img/cnn_curves.png")); display(Image("reports/img/cnn_confusion.png"))

## 6) المقارنة (accuracy / loss / الوقت)

In [ ]:
import argparse
results = [res_ann, res_cnn]
args = argparse.Namespace(img=IMG, gray=GRAY, epochs=EPOCHS, batch=BATCH)
ti.write_report(results, classes, len(tr), len(va), args, "reports/img/img_compare.md", by_video)
ti.compare_plot(results, "reports/img/img_compare.png")
display(Markdown(Path("reports/img/img_compare.md").read_text(encoding="utf-8")))
display(Image("reports/img/img_compare.png"))

## 7) صورك انت (test)

**أ) صوّر نفسك:** الخلية دي بتفتح الكاميرا (`0` أو `1` لكاميرا اللابتوب، `esp32` لكاميرا الكاب). اضغط `1` = alert، `2` = microsleep، `3` = yawning، `0` = pause، `q` = خروج. الصور بتتحفظ في `my_images\<class>`، على جهازك بس، ومتترفعش على git.

In [ ]:
CAMERA = "0"
!{sys.executable} src/capture_images.py --participant P01 --classes {" ".join(classes)} --camera {CAMERA}

**ب) اختبر الموديلين على الصور دي:**

In [ ]:
!{sys.executable} src/test_img.py --images my_images --models models/img_cnn.pt models/img_ann.pt
for n in ("cnn", "ann"):
    f = f"reports/img_test/test_{n}_confusion.png"
    if os.path.exists(f): display(Image(f))

الـ predictions لكل صورة في `reports/img_test/predictions_cnn.csv` و `predictions_ann.csv`.

**ملحوظة مهمة:** لو الـ accuracy على الـ validation عالية (95%+) وعلى صورك واطية، يبقى الداتا الجاهزة فيها نفس الأشخاص في التدريب والـ validation (الموديل حفظ الوشوش) أو شكل الكاميرا/الإضاءة مختلف. ده طبيعي، والحل إنك تضيف صورك في التدريب كمان (قولّي وأضيفهالك).